# Cells

This notebook uses a real PBMC AnnData example to show model-aware cell embedding preprocessing through `BioEmbedder.embed(...)`. It also demonstrates real cell-line annotation with `CellLineAnnotator` as a separate metadata workflow.



In [ ]:
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
from IPython.display import display

from embpy import BioEmbedder, pl, tl
from embpy.resources import CellLineAnnotator

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

embedder = BioEmbedder(device="auto", organism="human")


def compact_obs(adata: ad.AnnData, prefixes: tuple[str, ...], base: list[str] | None = None) -> pd.DataFrame:
    base = base or []
    cols = [c for c in base if c in adata.obs.columns]
    cols += [c for c in adata.obs.columns if c.startswith(prefixes)]
    return adata.obs.loc[:, list(dict.fromkeys(cols))]

adata = sc.datasets.pbmc3k()
adata.var_names_make_unique()
adata = adata[:500, :1000].copy()
adata



## Embed cells with the model-aware preprocessing policy

For `model="pca"`, `preprocessing="auto"` resolves to the standard single-cell pipeline: counts are preserved, normalized/log-transformed data are stored in `.layers["log_normalized"]`, and the PCA embedding is written to `.obsm`.



In [ ]:
cell_space = embedder.embed(
    adata,
    entity_type="cell",
    model="pca",
    output="anndata",
    preprocessing="auto",
    n_pca_components=8,
    pca_use_hvg=False,
    select_hvg=False,
    min_genes=0,
    min_cells=0,
    max_pct_mito=100.0,
    key="X_pca_auto",
)

display(cell_space)
print("obsm keys:", list(cell_space.obsm.keys()))
print("layers:", list(cell_space.layers.keys()))
print("preprocessing metadata:", cell_space.uns["embpy_cell_embeddings"]["__preprocessing__"])



## Compare another real preprocessing choice

This second call changes the normalization target before PCA. The model call is still real and records its own preprocessing metadata.



In [ ]:
cell_space = embedder.embed(
    cell_space,
    entity_type="cell",
    model="pca",
    output="anndata",
    preprocessing="standard",
    target_sum=5_000,
    n_pca_components=8,
    pca_use_hvg=False,
    select_hvg=False,
    min_genes=0,
    min_cells=0,
    max_pct_mito=100.0,
    key="X_pca_target5000",
)

print("obsm keys:", list(cell_space.obsm.keys()))
print("latest preprocessing metadata:", cell_space.uns["embpy_cell_embeddings"]["__preprocessing__"])



## Plot and compare cell embeddings



In [ ]:
color_key = "n_genes" if "n_genes" in cell_space.obs else None
pl.plot_embedding_space(
    cell_space,
    obsm_key="X_pca_auto",
    method="pca",
    color=color_key,
    title="PBMC PCA embedding from BioEmbedder.embed(preprocessing='auto')",
)

pl.plot_embedding_space(
    cell_space,
    obsm_key="X_pca_target5000",
    method="pca",
    color=color_key,
    title="PBMC PCA embedding with target_sum=5000",
)

k = min(10, cell_space.n_obs - 1)
_, mean_overlap = tl.compute_knn_overlap(cell_space, "X_pca_auto", "X_pca_target5000", k=k)
print(f"Mean PCA/PCA-target5000 KNN overlap: {mean_overlap:.3f}")
pl.knn_overlap(cell_space, obsm_keys=["X_pca_auto", "X_pca_target5000"], k=k)
pl.cross_embedding_correlation(cell_space, "X_pca_auto", "X_pca_target5000")



## Real cell-line annotation

Cell-line annotation is a metadata workflow, separate from PBMC cell-state embedding. The calls below query embpy's cell-line annotation resources and write results to `.obs` and `.uns`.



In [ ]:
cell_line_adata = ad.AnnData(
    X=np.zeros((4, 1), dtype=np.float32),
    obs=pd.DataFrame(
        {"cell_line": ["A549", "K562", "Jurkat", "THP-1"]},
        index=["A549", "K562", "Jurkat", "THP-1"],
    ),
)

annotator = CellLineAnnotator()
cell_line_adata = annotator.annotate_adata(
    cell_line_adata,
    column="cell_line",
    sources=["cellosaurus", "depmap", "passports"],
)

display(compact_obs(cell_line_adata, ("cellline_",), base=["cell_line"]))
print("annotation stores:", [k for k in cell_line_adata.uns if "annotation" in k])



## Save reusable artifacts



In [ ]:
cell_space.write_h5ad(OUTPUT_DIR / "pbmc_cell_embeddings.h5ad")
cell_line_adata.write_h5ad(OUTPUT_DIR / "cell_line_annotations.h5ad")
print(OUTPUT_DIR / "pbmc_cell_embeddings.h5ad")
print(OUTPUT_DIR / "cell_line_annotations.h5ad")

